In [0]:
fact_resident_feedback = spark.sql(f"select * from regis_healthcare.silver.resident_feedback;")
fact_resident_feedback.createOrReplaceTempView("resident_feedback")

In [0]:
# Fact_Resident_Feedback -- > Source: resident_feedback
# | Foreign Keys      |
# | ----------------- |
# | feedback_key      |
# | resident_key      |
# | facility_key      |
# | feedback_date_key |
from pyspark.sql.functions import regexp_replace,col,when
fact_resident_feedback = fact_resident_feedback.withColumn(
    "feedback_key",
    regexp_replace(col("feedback_id"), "^FBK", "").cast("int")
)
fact_resident_feedback = fact_resident_feedback.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
fact_resident_feedback = fact_resident_feedback.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)
from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_resident_feedback = fact_resident_feedback.withColumn("feedback_date_key", date_format(col("feedback_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_resident_feedback = fact_resident_feedback.withColumn("feedback_date_key", col("feedback_date_key").cast("int"))
#-------------------------
fact_resident_feedback = fact_resident_feedback.select(
 "feedback_key",      
 "resident_key",      
 "facility_key",      
 "feedback_date_key" )
display(fact_resident_feedback)


#### cataloge 

In [0]:
fact_resident_feedback.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_resident_feedback")

In [0]:
fact_resident_feedback.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_resident_feedback")
print(fact_resident_feedback.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_resident_feedback")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_resident_feedback")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.feedback_key = source.feedback_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_resident_feedback;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_resident_feedback;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_resident_feedback.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_resident_feedback")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_resident_feedback"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_resident_feedback

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.feedback_key = source.feedback_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
